# 4.1 VAE Imputation: Training

<div class="alert alert-info">

<strong>Ziel</strong><br>
Training von Variational Autoencoders (VAE) auf <strong>vollständigen Datenzeilen</strong>.

Es werden wie bereits zuvor in 3.2 verschiedene Feature-Kombinationen durchiteriert
<br><br>
<strong>Vorgehen</strong>
<ol>
    <li><strong>Daten laden</strong>: Aus dem neuesten Preprocessing-Ordner (3.1)</li>
    <li><strong>Feature-Auswahl</strong>: Iteration über Basis-Features + Optionale Features (bis zu 30 Kombinationen aktuell, anpassbar)</li>
    <li><strong>Filterung</strong>: Nutzung ausschließlich vollständiger Datensätze für die gewählten Spalten</li>
    <li><strong>Training</strong>: Ein VAE pro Kombination wird trainiert und gespeichert</li>
    <li><strong>Output</strong>: Gespeicherte Modelle im <code>Models/</code> Unterordner für spätere Inferenz (4.2)</li>
</ol>
</div>

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import os
import itertools
import joblib

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from pathlib import Path
from datetime import datetime

# Reproduzierbarkeit
torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid")

In [2]:
# ------------------------- Konfiguration -------------------------

FIXED_BASE_FEATURES = [
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
]

OPTIONAL_FEATURES_POOL = [
    # ------------------------- Physikalische Parameter -------------------------
    "temperature_in_c",
    "pH",
    "electrical_conductivity_25c_in_uS/cm",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",

    # ------------------------- Weitere Ionen / Spurenelemente -------------------------
    "K_in_mmol/L",
    "NO3_in_mmol/L",
    "Li_in_mmol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "HS_in_mmol/L",
    "O2_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "F_in_umol/L",
    "H2SiO3_in_umol/L",
    
    # ------------------------- Metadaten (nur numerisch nutzen) -------------------------
    "depth_bgl_in_m",
    # "stratigraphic_period",
    # "rock_type"
]

MAX_ITERATIONS = 30
BATCH_SIZE = 64
EPOCHS = 150  # Increased for better convergence
LATENT_DIM = 4
KLD_WARMUP_EPOCHS = 50 # Annealing duration

In [3]:
# ------------------------- Daten Laden Helper -------------------------
base_dir = Path.cwd()


# ------------------------- Suche nach dem neuesten Preprocessing-Ordner -------------------------
preprocessing_root = base_dir.parent.parent / "3_Machine-Learning" / "3.1_Preprocessing" / "Preprocessing"
timestamp_folders = [f for f in preprocessing_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Preprocessing-Ordner in {preprocessing_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Verwendeter Preprocessing-Ordner: {latest_folder.name}")

input_path_prep = latest_folder / "Preprocessed_SOM_Ready.csv"
df_full = pd.read_csv(input_path_prep, low_memory=False)
print(f"Daten geladen. Shape: {df_full.shape}")

Verwendeter Preprocessing-Ordner: 2026-01-09_14-06-57


Daten geladen. Shape: (94264, 50)


In [4]:
# ------------------------- Feature Resolution Helper -------------------------
def get_training_features(user_selection, df_columns):
    training_features = []
    mapping_report = {}
    
    for user_col in user_selection:
        found = False
        candidates = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
        if candidates:

            # ------------------------- Priorisiere _log_gauss -------------------------
            best_match = next((c for c in candidates if "_log_gauss" in c), candidates[0])
            training_features.append(best_match)
            mapping_report[user_col] = best_match
            found = True
        else:
            if user_col in df_columns:
                training_features.append(user_col)
                mapping_report[user_col] = f"{user_col} (Raw)"
                found = True
        
        if not found:
             # ------------------------- Fallback -------------------------
             fuzzy = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
             if fuzzy:
                 best_match = fuzzy[0]
                 training_features.append(best_match)
                 mapping_report[user_col] = f"{best_match} (Fuzzy)"
                 found = True
                 
        if not found:
             print(f"Warnung: Feature '{user_col}' nicht gefunden!")
             
    return training_features, mapping_report

In [5]:
# ------------------------- VAE Model Definition -------------------------
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=4, hidden_dim=32):
        super(VAE, self).__init__()
        
        # ------------------------- Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # ------------------------- Decoder
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
        
        self.relu = nn.ReLU()

    def encode(self, x):
        h1 = self.relu(self.fc1(x))
        return self.fc2_mu(h1), self.fc2_logvar(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z):
        h3 = self.relu(self.fc3(z))
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# -------------------------------------------------- Verlustfunktion --------------------------------------------------
# Wird nur von den Daten selbst generiert, kein spezielles Label, von dem das Modell abhängig ist

def loss_function(recon_x, x, mu, logvar, beta=1.0):
    MSE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return MSE + (beta * KLD)

In [ ]:
# ------------------------- Ordner für Modelle -------------------------
models_dir = base_dir / "Models" / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
models_dir.mkdir(parents=True, exist_ok=True)
print(f"Modelle werden gespeichert in: {models_dir}")

# ------------------------- Trainings Loop -------------------------
run_counter = 0
history_log = []

# ------------------------- Basis Run -------------------------
combinations = [ [] ] # Leere Liste = Nur Hauptionen

# ------------------------- Zusatz Features Iterationen -------------------------
for r in range(1, 4):
    combinations.extend(list(itertools.combinations(OPTIONAL_FEATURES_POOL, r)))


print(f"Geplante Kombinationen (Gesamt): {len(combinations)}")
print(f"Ausführung limitiert auf: {MAX_ITERATIONS}")

for i, combo in enumerate(combinations):
    if run_counter >= MAX_ITERATIONS:
        break
        
    run_counter += 1
    run_id = f"Run_{run_counter:03d}"
    

    # ------------------------- Features zusammenstellen -------------------------
    current_selection = FIXED_BASE_FEATURES + list(combo)
    
    # ------------------------- Mapping auf Spalten -------------------------
    train_cols, mapping = get_training_features(current_selection, df_full.columns)
    

    # ------------------------- Daten Filtern (Nur vollständige Zeilen) -------------------------
    df_run = df_full[train_cols].dropna().copy()
    
    n_samples = len(df_run)
    if n_samples < 100:
        print(f"[{run_id}] Zu wenige Daten ({n_samples}). Skip.")
        continue
        
    print(f"\n--- {run_id} | Features: {len(train_cols)} | Samples: {n_samples} ---")
    print(f"Combo: {combo}")


    # ------------------------- Skalierung mit StandardScaler -------------------------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_run.values)
    
    # ------------------------- Tensor Datensatz -------------------------
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_scaled).float()), batch_size=BATCH_SIZE, shuffle=True)
    

    # ------------------------- Modell Initialisierung -------------------------
    input_dim = len(train_cols)
    vae = VAE(input_dim=input_dim, latent_dim=LATENT_DIM)
    optimizer = optim.Adam(vae.parameters(), lr=1e-3)
    

    # ------------------------- Training -------------------------
    vae.train()
    final_loss = 0
    for epoch in range(EPOCHS):
        # KLD Annealing / Warmup
        if epoch < KLD_WARMUP_EPOCHS:
            current_beta = epoch / KLD_WARMUP_EPOCHS
        else:
            current_beta = 1.0
        epoch_loss = 0
        for batch in train_loader:
            x_batch = batch[0]
            optimizer.zero_grad()
            recon_x, mu, logvar = vae(x_batch)
            loss = loss_function(recon_x, x_batch, mu, logvar, beta=current_beta)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if epoch == EPOCHS - 1:
            final_loss = epoch_loss / n_samples
            
    print(f"  -> Final Loss: {final_loss:.4f}")
    

    # ------------------------- Speichern -------------------------
    # Jeweils mit gleichem Run_XYZ Prefix
    model_path = models_dir / f"{run_id}_vae.pth"
    scaler_path = models_dir / f"{run_id}_scaler.joblib"
    meta_path = models_dir / f"{run_id}_meta.json"
    
    torch.save(vae.state_dict(), model_path)
    joblib.dump(scaler, scaler_path)
    

    # ------------------------- Metadaten speichern für Inferenz -------------------------
    metadata = {
        "run_id": run_id,
        "features_user": current_selection,
        "features_mapped": train_cols,
        "mapping_report": mapping,
        "n_samples": n_samples,
        "final_loss": final_loss,
        "latent_dim": LATENT_DIM
    }
    import json
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=4)
        
    history_log.append(metadata)

print("\nTraining beendet")

Modelle werden gespeichert in: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.1_VAE_Imputation\Models\2026-01-09_14-27-15
Geplante Kombinationen (Gesamt): 834
Ausführung limitiert auf: 30

--- Run_001 | Features: 6 | Samples: 75016 ---
Combo: []


  -> Final Loss: 4.1589

--- Run_002 | Features: 7 | Samples: 5525 ---
Combo: ('temperature_in_c',)


  -> Final Loss: 4.7528

--- Run_003 | Features: 7 | Samples: 60848 ---
Combo: ('pH',)


  -> Final Loss: 4.8451

--- Run_004 | Features: 7 | Samples: 1219 ---
Combo: ('electrical_conductivity_25c_in_uS/cm',)


  -> Final Loss: 4.7340

--- Run_005 | Features: 7 | Samples: 319 ---
Combo: ('redox_potential_in_mV',)


...

  -> Final Loss: 5.4135

--- Run_026 | Features: 8 | Samples: 3151 ---
Combo: ('temperature_in_c', 'Fe_in_mmol/L')


  -> Final Loss: 5.4163

--- Run_027 | Features: 8 | Samples: 1630 ---
Combo: ('temperature_in_c', 'Mn_in_mmol/L')


  -> Final Loss: 5.4880
[Run_028] Zu wenige Daten (1). Skip.

--- Run_029 | Features: 8 | Samples: 235 ---
Combo: ('temperature_in_c', 'O2_in_mmol/L')


  -> Final Loss: 5.3528

--- Run_030 | Features: 8 | Samples: 1377 ---
Combo: ('temperature_in_c', 'Sr_in_umol/L')


  -> Final Loss: 5.3777

Training beendet


In [7]:

# ------------------------- SIGNAL DONE -------------------------
signal_file = models_dir / "DONE_TRAINING"
signal_file.touch()
print(f"Signal file created: {signal_file.name}")


Signal file created: DONE_TRAINING
